# BioCirv AI Stable Prototype (v2)

This Colab workflow follows the stability plan in plans/biocirv_ai_stable_prototype_plan.md.

**Objectives**
- Query `ca_biositing` and `data_portal` materialized views with natural language.
- Use sandbox_setup_v2 for proper ColumnSchema metadata.
- Run in Google Colab with Cloud SQL Proxy, CBORG gateway, and GCP Secret Manager.

> Run each cell sequentially; cells are idempotent when the Cloud SQL Proxy PID is tracked in the environment.

In [ ]:
# Cell 1 — Environment Bootstrap
# Requires a GitHub PAT stored in Colab Secrets as GITHUB_TOKEN
# (Colab → 🔑 Secrets panel → add GITHUB_TOKEN with repo read scope)
import os
import sys
import pathlib
import subprocess

# ── 1. Resolve GitHub token (private repo auth) ──────────────────────────────
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '')

if not GITHUB_TOKEN:
    raise RuntimeError(
        'GITHUB_TOKEN not found. '
        'Add it in Colab → 🔑 Secrets with repo read scope.'
    )

# Embed token in URL so git never prompts for credentials
REPO_OWNER = 'pjsmitty301'
REPO_NAME  = 'ca-biositing'
AUTH_URL   = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'

# ── 2. Clone or update the repository ────────────────────────────────────────
PROJECT_ROOT = pathlib.Path('/content/ca-biositing')

if not PROJECT_ROOT.exists():
    print(f'📦 Cloning {REPO_OWNER}/{REPO_NAME} ...')
    subprocess.run(
        ['git', 'clone', '--depth=1', AUTH_URL, str(PROJECT_ROOT)],
        check=True
    )
else:
    print('📦 Repository exists — pulling latest changes...')
    # Re-set the remote URL with the token so pull is authenticated
    subprocess.run(
        ['git', '-C', str(PROJECT_ROOT), 'remote', 'set-url', 'origin', AUTH_URL],
        check=True
    )
    subprocess.run(
        ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'],
        check=True
    )

# ── 3. Install pinned Colab dependencies ─────────────────────────────────────
requirements = PROJECT_ROOT / 'analysis/biocirv-ai/requirements-colab.txt'
if not requirements.exists():
    raise FileNotFoundError(f'requirements-colab.txt not found at {requirements}')

print(f'📚 Installing pinned dependencies from {requirements} ...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)],
    check=True
)

# ── 4. Add src/ to Python path ────────────────────────────────────────────────
SRC_PATH = PROJECT_ROOT / 'analysis/biocirv-ai/src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print('✅ Environment bootstrap complete.')


In [ ]:
import os
import pathlib
import subprocess
import time
from typing import Optional

PROJECT_ROOT = pathlib.Path('/content/ca-biositing')
PROJECT_ID = os.environ.get('GCP_PROJECT_ID', 'biocirv-470318')
INSTANCE_CONNECTION_NAME = os.environ.get('INSTANCE_CONNECTION_NAME', 'biocirv-470318:us-central1:biocirv-staging')
DB_PORT = os.environ.get('DB_PORT', '5434')

print(f'GCP project: {PROJECT_ID}')
print(f'Cloud SQL instance: {INSTANCE_CONNECTION_NAME}')

try:
    from google.colab import auth
    auth.authenticate_user()
    print('✅ Google auth complete.')
except Exception as exc:
    print(f'⚠️ Google auth skipped: {exc}')

from google.cloud import secretmanager
client = secretmanager.SecretManagerServiceClient()

def fetch_secret(secret_id: Optional[str]):
    if not secret_id:
        return None
    name = f'projects/{PROJECT_ID}/secrets/{secret_id}/versions/latest'
    try:
        response = client.access_secret_version(request={'name': name})
        return response.payload.data.decode('utf-8').strip()
    except Exception as exc:
        print(f"⚠️ Secret '{secret_id}' unavailable: {exc}")
        return None

env_updates = {
    'CBORG_API_KEY': os.environ.get('CBORG_API_KEY') or fetch_secret(os.environ.get('CBORG_SECRET', 'CBORG_API_KEY')),
    'DB_USER': os.environ.get('DB_USER') or fetch_secret(os.environ.get('DB_USER_SECRET', 'biocirv-readonly-username')) or 'biocirv_readonly',
    'DB_PASS': os.environ.get('DB_PASS') or fetch_secret(os.environ.get('DB_PASS_SECRET', 'biocirv-readonly-password')),
    'DB_NAME': os.environ.get('DB_NAME', 'biocirv-staging'),
    'DB_HOST': '127.0.0.1',
    'DB_PORT': DB_PORT,
    'CLOUD_MODE': 'true',
}

missing = []
for key, value in env_updates.items():
    if value:
        os.environ[key] = value
    else:
        missing.append(key)

if missing:
    raise RuntimeError(f'Missing required secrets: {missing}')

proxy_path = pathlib.Path('/content/cloud-sql-proxy')
if not proxy_path.exists():
    print('⬇️ Downloading Cloud SQL Proxy binary...')
    subprocess.run([
        'curl', '-o', str(proxy_path),
        'https://storage.googleapis.com/cloud-sql-connectors/cloud-sql-proxy/v2.11.2/cloud-sql-proxy.linux.amd64'
    ], check=True)
    proxy_path.chmod(0o755)

if os.environ.get('CLOUD_SQL_PROXY_PID'):
    print(f"ℹ️ Proxy already running (pid={os.environ['CLOUD_SQL_PROXY_PID']}).")
else:
    log_file = open('/content/cloud-sql-proxy.log', 'w')
    proxy_cmd = [
        str(proxy_path),
        INSTANCE_CONNECTION_NAME,
        f'--port={DB_PORT}',
        '--quiet'
    ]
    print('🚀 Starting Cloud SQL Proxy:', ' '.join(proxy_cmd))
    proc = subprocess.Popen(proxy_cmd, stdout=log_file, stderr=subprocess.STDOUT)
    time.sleep(5)
    os.environ['CLOUD_SQL_PROXY_PID'] = str(proc.pid)
    print(f'✅ Cloud SQL Proxy running (pid={proc.pid}). Logs → /content/cloud-sql-proxy.log')

env_file = PROJECT_ROOT / 'analysis/biocirv-ai/.env.colab'
with env_file.open('w') as fh:
    for key in ['CBORG_API_KEY', 'DB_USER', 'DB_PASS', 'DB_NAME', 'DB_HOST', 'DB_PORT', 'CLOUD_MODE']:
        fh.write(f'{key}={os.environ[key]}
')
print(f'📝 Persisted runtime env vars to {env_file}')


In [ ]:
import os
import pandas as pd
import sqlalchemy as sa
from sqlalchemy import text

engine = sa.create_engine(
    f"postgresql+psycopg2://{os.environ['DB_USER']}:{os.environ['DB_PASS']}@"
    f"{os.environ['DB_HOST']}:{os.environ['DB_PORT']}/{os.environ['DB_NAME']}"
)

with engine.connect() as conn:
    version = conn.execute(text('select version()'))
    view_count = conn.execute(text('select count(*) from information_schema.views where table_schema in ('ca_biositing','data_portal')'))
print('✅ Connected to PostgreSQL')
print(version.scalar_one())
print(f'Total materialized + regular views detected: {view_count.scalar_one()}')

sample = pd.read_sql_query(
    'SELECT * FROM "ca_biositing"."analysis_average_view" LIMIT 5',
    con=engine
)
sample


In [ ]:
import pandas as pd
from ca_biositing.ai_exploration.schema import discover_views, fetch_table_metadata

views = discover_views(engine, ['ca_biositing', 'data_portal'])
rows = []
for view in views:
    rows.append({
        'schema': view['schema'],
        'view': view['table'],
        'columns': fetch_table_metadata(engine, view['table'], schema=view['schema'])
    })

display(pd.DataFrame(rows))


In [ ]:
import os
from ca_biositing.ai_exploration.sandbox_setup_v2 import init_agent

TARGET_VIEWS = [
    'ca_biositing.analysis_data_view',
    'ca_biositing.analysis_average_view',
    'data_portal.mv_biomass_availability',
    'data_portal.mv_biomass_composition',
    'data_portal.mv_biomass_sample_stats',
    'data_portal.mv_biomass_fermentation',
]

agent = init_agent(
    model_name=os.environ.get('CBORG_MODEL', 'gemini-3-flash'),
    cloud_mode=True,
    qualified_views=TARGET_VIEWS
)
agent


In [ ]:
result = agent.chat('Show me the top 10 resources by average moisture content from ca_biositing.analysis_average_view.')
result.display()

In [ ]:
result = agent.chat('Create a bar chart of biomass availability by county using data_portal.mv_biomass_availability.')
result.display()

In [ ]:
result = agent.chat('Compare fermentation yields across pretreatment methods using data_portal.mv_biomass_fermentation.')
result.display()

In [ ]:
prompt = 'List the counties with the highest biomass moisture variability.'
result = agent.chat(prompt)
result.display()